# Bronze Câmbio BCB

## Objetivo
Ingerir a cotação do dólar comercial via API PTAX do Banco Central do Brasil, trazendo a cotação de venda e calculando a média diária.

## Estratégia adotada
A API PTAX possui recursos por período e por data. Como a consulta por período está retornando erro 500 no ambiente atual, a coleta será feita por data, usando o recurso oficial `CotacaoDolarDia`.

## Regra de negócio
- custo em USD por produto
- taxa de câmbio da data da venda
- uso da **média da cotação de venda do dia**

## Saída esperada
Tabela Delta:
`lh_nautical.02_bronze.cambio_bronze`

In [0]:
# ======================================================================================
# IMPORTS
# ======================================================================================

import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

## 1. Função de coleta por dia

Será utilizado o recurso oficial `CotacaoDolarDia`, consultando uma data por vez.
Pois o endpoint por período está instável.

In [0]:
# ======================================================================================
# FUNÇÃO DE COLETA - COTAÇÃO DO DÓLAR EM UMA DATA ESPECÍFICA
# ======================================================================================

def fetch_ptax_day(target_date: str) -> pd.DataFrame:
    """
    Consulta a API PTAX do Banco Central para uma data específica.

    Parâmetros
    ----------
    target_date : str
        Data no formato MM-DD-YYYY, exigido pelo parâmetro cotacaoDolarDia.

    Retorno
    -------
    pd.DataFrame
        DataFrame pandas com os registros retornados pela API para a data informada.
        Caso a data não tenha cotação disponível, retorna DataFrame vazio.
    """

    url = (
        "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        f"CotacaoDolarDia(dataCotacao=@dataCotacao)?@dataCotacao='{target_date}'&$format=json"
    )

    response = requests.get(url, timeout=60)

    # Em caso de erro do servidor, retornamos DataFrame vazio para não quebrar o pipeline.
    if response.status_code != 200:
        return pd.DataFrame()

    payload = response.json()
    return pd.DataFrame(payload.get("value", []))

## 2. Definição do intervalo de coleta

Como a base de vendas cobre 2023 e 2024, a coleta será feita para esse período, ja considerando a necessidade de expandir para dezembro/2022, já que o primeiro da nossa base é domingo, e precisa de cotação do dia útil anterior.

In [0]:
# ======================================================================================
# INTERVALO DE COLETA
# ======================================================================================


start_date = datetime(2022, 12, 1)
end_date = datetime(2024, 12, 31)

url = f"...start={start_date.strftime('%Y-%m-%d')}&end={end_date.strftime('%Y-%m-%d')}"

## 3. Coleta diária

A coleta percorre o intervalo de datas e consulta a API um dia por vez.

Observação:
nem toda data terá cotação disponível, por exemplo:
- fins de semana
- feriados

In [0]:
# ======================================================================================
# COLETA DIÁRIA DA PTAX
# ======================================================================================

dfs = []
current_date = start_date

while current_date <= end_date:
    # A API trabalha com data no formato MM-DD-YYYY
    api_date = current_date.strftime("%m-%d-%Y")

    df_day = fetch_ptax_day(api_date)

    if not df_day.empty:
        dfs.append(df_day)

    current_date += timedelta(days=1)

print(f"Quantidade de DataFrames coletados: {len(dfs)}")

Quantidade de DataFrames coletados: 524


In [0]:
# ======================================================================================
# UNIÃO DOS DADOS COLETADOS
# ======================================================================================

if dfs:
    df_cambio_pd = pd.concat(dfs, ignore_index=True)
else:
    df_cambio_pd = pd.DataFrame()

print("Quantidade total de linhas coletadas:", df_cambio_pd.shape[0])
df_cambio_pd.head()

Quantidade total de linhas coletadas: 524


,cotacaoCompra,cotacaoVenda,dataHoraCotacao
0,5.1942,5.1948,2022-12-01 13:09:56.009
1,5.1968,5.1974,2022-12-02 13:03:46.581
2,5.2457,5.2464,2022-12-05 13:09:22.368
3,5.2390,5.2396,2022-12-06 13:03:41.87
4,5.2202,5.2208,2022-12-07 13:05:19.208


## 4. Conversão para Spark DataFrame

A partir daqui, o processamento segue o padrão do projeto no Databricks.

In [0]:
# ======================================================================================
# CONVERSÃO PARA SPARK
# ======================================================================================

df_cambio_raw = spark.createDataFrame(df_cambio_pd)

display(df_cambio_raw.limit(10))

cotacaoCompra,cotacaoVenda,dataHoraCotacao
5.1942,5.1948,2022-12-01 13:09:56.009
5.1968,5.1974,2022-12-02 13:03:46.581
5.2457,5.2464,2022-12-05 13:09:22.368
5.239,5.2396,2022-12-06 13:03:41.87
5.2202,5.2208,2022-12-07 13:05:19.208
5.2085,5.2091,2022-12-08 13:02:26.317
5.239,5.2396,2022-12-09 13:09:49.371
5.306,5.3066,2022-12-12 13:09:23.408
5.2663,5.2669,2022-12-13 13:05:48.922
5.34,5.3406,2022-12-14 13:02:32.928


## 5. Transformação da camada bronze

Objetivos:
- transformar `dataHoraCotacao` em data
- converter `cotacaoVenda` para numérico
- calcular a média diária da cotação de venda

In [0]:
# ======================================================================================
# TRANSFORMAÇÃO - CÂMBIO BRONZE
# ======================================================================================

df_cambio_bronze = (
    df_cambio_raw
    .select(
        F.to_date(F.col("dataHoraCotacao")).alias("exchange_date"),
        F.col("cotacaoVenda").cast(DoubleType()).alias("exchange_rate")
    )
    .dropna(subset=["exchange_date", "exchange_rate"])
    .groupBy("exchange_date")
    .agg(
        F.avg("exchange_rate").alias("exchange_rate")
    )
    .orderBy("exchange_date")
)

display(df_cambio_bronze.limit(10))

exchange_date,exchange_rate
2022-12-01,5.1948
2022-12-02,5.1974
2022-12-05,5.2464
2022-12-06,5.2396
2022-12-07,5.2208
2022-12-08,5.2091
2022-12-09,5.2396
2022-12-12,5.3066
2022-12-13,5.2669
2022-12-14,5.3406


## 6. Validações

Validações esperadas:
- schema correto
- datas cobrindo o intervalo útil
- ausência de nulos críticos

In [0]:
# ======================================================================================
# VALIDAÇÕES
# ======================================================================================

df_cambio_bronze.printSchema()

print("Quantidade de dias com cotação:", df_cambio_bronze.count())

display(
    df_cambio_bronze.agg(
        F.min("exchange_date").alias("min_date"),
        F.max("exchange_date").alias("max_date"),
        F.min("exchange_rate").alias("min_rate"),
        F.max("exchange_rate").alias("max_rate")
    )
)

display(
    df_cambio_bronze.filter(
        F.col("exchange_date").isNull() | F.col("exchange_rate").isNull()
    )
)

root
 |-- exchange_date: date (nullable = true)
 |-- exchange_rate: double (nullable = true)

Quantidade de dias com cotação: 524


min_date,max_date,min_rate,max_rate
2022-12-01,2024-12-31,4.7202,6.1991


exchange_date,exchange_rate


## 7. Persistência em Delta

A tabela será salva na camada bronze para ser consumida no dbt posteriormente.

In [0]:
# ======================================================================================
# ESCRITA - CÂMBIO BRONZE
# ======================================================================================

spark.sql("DROP TABLE IF EXISTS lh_nautical.02_bronze.cambio_bronze")

df_cambio_bronze.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.02_bronze.cambio_bronze")

In [0]:
display(
    spark.table("lh_nautical.02_bronze.cambio_bronze").limit(10)
)

exchange_date,exchange_rate
2022-12-01,5.1948
2022-12-02,5.1974
2022-12-05,5.2464
2022-12-06,5.2396
2022-12-07,5.2208
2022-12-08,5.2091
2022-12-09,5.2396
2022-12-12,5.3066
2022-12-13,5.2669
2022-12-14,5.3406


In [0]:
# Cruzando dados para confirmar se todas as datas tem cotaçao

df_datas_venda = (
    spark.table("lh_nautical.02_bronze.vendas_bronze")
    .select("sale_date")
    .distinct()
)

df_datas_cambio = (
    spark.table("lh_nautical.02_bronze.cambio_bronze")
    .select(F.col("exchange_date"))
    .distinct()
)

datas_sem_cambio = (
    df_datas_venda.join(
        df_datas_cambio,
        df_datas_venda.sale_date == df_datas_cambio.exchange_date,
        "left_anti"
    )
)

display(datas_sem_cambio.orderBy("sale_date"))

sale_date
2023-01-01
2023-01-07
2023-01-08
2023-01-14
2023-01-21
2023-01-22
2023-01-28
2023-01-29
2023-02-04
2023-02-05


## Conclusão

A tabela `lh_nautical.02_bronze.cambio_bronze` foi construída contendo a cotação média diária de venda do dólar (PTAX) para o período analisado.

Nesta camada, os dados foram mantidos fiéis à fonte original, contemplando apenas os dias em que há publicação oficial de cotação pelo Banco Central. Dessa forma, não foi realizada nenhuma interpolação ou preenchimento de datas ausentes (como finais de semana e feriados).

Essa decisão segue o princípio da arquitetura de dados em camadas (Medallion), onde a camada Bronze deve preservar os dados em seu estado mais próximo possível da origem, aplicando apenas transformações técnicas.

As regras de negócio necessárias para a análise, como a utilização da última cotação disponível para datas sem publicação — serão implementadas na camada Silver (dbt).